# CUAD Fine-tune — extractive-QA on 41 clause types (Colab GPU)

Runs on Colab with a GPU runtime. Fine-tunes a DeBERTa-v3-base extractive-QA model on
the CUAD dataset, tracks to a local MLflow file-store, saves a self-contained bundle,
and registers it as `cuad-extractor` in the MLflow model registry.

**Runtime:** `Runtime > Change runtime type > GPU (T4 or better)`.

## 1. Install

Install the package from the merged `master` branch (or swap in your feature branch name).

In [ ]:
# Use a GPU runtime: Runtime > Change runtime type > GPU.
# --no-cache-dir avoids pip serving a stale cached build (the version stays 0.1.0).
# Do NOT use --force-reinstall here: it drags every dependency (numpy/torch/...) and
# clashes with Colab's preinstalled stack, leaving mismatched, broken installs.
!pip install -q --no-cache-dir "docintel[train,kie] @ git+https://github.com/KhoiDang1209/AI-Document-Understanding.git@master#subdirectory=docintel"
# Colab's torchaudio is built for a different CUDA than its torch; QA training doesn't use it, so drop it.
!pip uninstall -y -q torchaudio

## 2. Setup

In [ ]:
import subprocess
from pathlib import Path

import mlflow

from docintel.contracts.qa_config import QaTrainingConfig
from docintel.contracts.train_qa import run_qa_training

# 1 epoch + batch 32 + bf16 (config default) keeps the run within a Colab Pro session.
config = QaTrainingConfig(num_train_epochs=1.0, train_batch_size=32)
mlflow.set_tracking_uri("file:./mlruns")  # local file-store on the Colab VM
mlflow.set_experiment("cuad-qa")

In [ ]:
# Checkpoint fallback: save to Google Drive so a timed-out session can resume.
# Mount Drive, reuse the ocr-checkpoints folder if it exists, otherwise create it.
# run_qa_training() auto-detects the latest checkpoint here and resumes from it.
from google.colab import drive

drive.mount("/content/drive")
CHECKPOINT_DIR = "/content/drive/MyDrive/ocr-checkpoints"
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print("Checkpoints ->", CHECKPOINT_DIR)

## 3. Load dataset

In [ ]:
from datasets import load_dataset

DATASET_REVISION = "main"  # pin for reproducibility
# trust_remote_code=True: CUAD ships a loader script (datasets 3.x requires opting in).
raw = load_dataset("theatticusproject/cuad-qa", revision=DATASET_REVISION, trust_remote_code=True)
print(raw)

## 4. Tokenize (SQuAD-style sliding window)

CUAD contracts are long documents. We use `doc_stride` and `max_seq_length` from
`QaTrainingConfig` (defaults: `stride=128`, `max_seq_length=512`) with
`return_overflowing_tokens=True` so each contract is split into overlapping windows.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.model_name)


def _tokenize_split(examples):
    """Tokenize (question, context) pairs with sliding-window overflow.

    Answer char-spans are mapped to token start/end indices using sequence_ids
    to restrict the search to context tokens. Windows whose context does not
    fully contain the answer (and no-answer questions) are labelled with the CLS
    index, the standard SQuAD-style impossible-answer target.
    """
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=config.max_seq_length,
        stride=config.doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    cls_index = 0  # impossible / no-answer target
    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        sequence_ids = tokenized.sequence_ids(i)
        answers = examples["answers"][sample_map[i]]
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue
        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(answers["text"][0])

        # Token span covering the context (sequence_id == 1) in this window.
        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1
        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1

        # Answer not fully inside this window's context -> CLS.
        if not (offsets[context_start][0] <= answer_start and offsets[context_end][1] >= answer_end):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        token_start = context_start
        while token_start <= context_end and offsets[token_start][0] <= answer_start:
            token_start += 1
        start_positions.append(token_start - 1)

        token_end = context_end
        while token_end >= context_start and offsets[token_end][1] >= answer_end:
            token_end -= 1
        end_positions.append(token_end + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


train_ds = raw["train"].map(_tokenize_split, batched=True, remove_columns=raw["train"].column_names)
eval_ds = raw["test"].map(_tokenize_split, batched=True, remove_columns=raw["test"].column_names)
print(f"train windows: {len(train_ds)}, eval windows: {len(eval_ds)}")

In [ ]:
# Sanity check (CPU, no GPU): verify labels before spending a GPU session.
# 1) What fraction of windows are CLS/(0,0)? Expect high (CUAD is sparse) but < 1.0.
# 2) For a sample of positive windows, the decoded span must match the gold answer.
n = len(train_ds)
cls_count = sum(1 for s, e in zip(train_ds["start_positions"], train_ds["end_positions"]) if s == 0 and e == 0)
print(f"CLS/(0,0) windows: {cls_count}/{n} = {cls_count / n:.3%}")

checked = mismatches = 0
for ex in train_ds:
    s, e = ex["start_positions"], ex["end_positions"]
    if s == 0 and e == 0:
        continue
    decoded = tokenizer.decode(ex["input_ids"][s : e + 1]).strip()
    if decoded:  # decoded span should be non-empty context text
        checked += 1
    else:
        mismatches += 1
    if checked >= 20:
        break
print(f"positive spans decoded non-empty: {checked} (empty: {mismatches})")
assert cls_count < n, "ALL windows are CLS -> labels collapsed; do not train."


## 5. Train & track

In [ ]:
git_sha = (
    subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True
    ).stdout.strip()
    or "unknown"
)
bundle = run_qa_training(
    config=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    bundle_dir=Path("cuad-extractor-bundle"),
    dataset_revision=DATASET_REVISION,
    git_sha=git_sha,
    output_dir=CHECKPOINT_DIR,  # checkpoints to Drive; resumes here after a timeout
)
print("Bundle saved to:", bundle)

## 6. Register in MLflow model registry

In [ ]:
# The last run logged the bundle as an artifact; register it.
runs = mlflow.search_runs(experiment_names=["cuad-qa"], order_by=["start_time DESC"])
run_id = runs.iloc[0]["run_id"]
artifact_uri = f"runs:/{run_id}/bundle"
mv = mlflow.register_model(artifact_uri, "cuad-extractor")
print(f"Registered cuad-extractor version {mv.version}")

## 7. Next step

Run `notebooks/cuad_onnx_export.ipynb` to download this bundle, export to ONNX fp32,
quantize to INT8, evaluate, and register `cuad-extractor-onnx-int8`.

To serve locally on the laptop set:
```
DOCINTEL_CONTRACT_ONNX_LOCAL_PATH=/path/to/cuad-int8-bundle
```
then start the API and `POST /contracts/extract` with a contract PDF.